# Explore Demographic Simulation Results

1. Variable presence across GSS years (2021, 2022, 2024) — which demographic/lifestyle variables are available in all three years?
2. Correlation analysis on public_issues results (from the running `run_demo_sim.py` job)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 60)

## Part 1: Variable Presence Across GSS Years

In [ ]:
# Load GSS data
df_gss = pd.read_csv('/project/jevans/maxzhuyt/gss-depth/gss_2021_2024.csv', low_memory=False)
print(f'GSS data: {df_gss.shape[0]} respondents, {df_gss.shape[1]} variables')
print(f'Years: {sorted(df_gss["year"].unique())}')
print(f'Respondents per year:')
print(df_gss['year'].value_counts().sort_index())

In [ ]:
# Load variable lists
demo_df = pd.read_csv('gss_question_lists/gss_demographic_variables.csv')
lifestyle_df = pd.read_csv('gss_question_lists/gss_politicized_lifestyle_variables.csv')

demo_vars = list(demo_df['VariableName'])
lifestyle_vars = list(lifestyle_df['VariableName'])

# Add polviews (not in demographic CSV)
if 'polviews' not in demo_vars:
    demo_vars.append('polviews')

print(f'Demographic variables: {len(demo_vars)}')
print(f'Lifestyle variables: {len(lifestyle_vars)}')
print(f'Overlap: {len(set(demo_vars) & set(lifestyle_vars))}')

In [ ]:
# For each variable, compute: (a) whether it exists in the GSS columns,
# (b) non-missing rate per year, (c) present in all 3 years?
years = sorted(df_gss['year'].unique())

def analyze_variable_presence(var_list, var_type_label):
    rows = []
    for var in var_list:
        row = {'variable': var, 'type': var_type_label}
        if var not in df_gss.columns:
            row['in_gss'] = False
            for y in years:
                row[f'pct_{y}'] = 0.0
                row[f'n_{y}'] = 0
            row['present_all_years'] = False
            row['n_years_present'] = 0
        else:
            row['in_gss'] = True
            n_years = 0
            for y in years:
                mask = df_gss['year'] == y
                n_total = mask.sum()
                n_valid = df_gss.loc[mask, var].notna().sum()
                row[f'pct_{y}'] = round(100 * n_valid / n_total, 1) if n_total > 0 else 0.0
                row[f'n_{y}'] = int(n_valid)
                if n_valid > 0:
                    n_years += 1
            row['present_all_years'] = (n_years == len(years))
            row['n_years_present'] = n_years
        rows.append(row)
    return pd.DataFrame(rows)

df_demo_presence = analyze_variable_presence(demo_vars, 'demographic')
df_life_presence = analyze_variable_presence(lifestyle_vars, 'lifestyle')
df_presence = pd.concat([df_demo_presence, df_life_presence], ignore_index=True)

# Drop duplicates (some vars appear in both lists)
df_presence = df_presence.drop_duplicates(subset='variable', keep='first')

print(f'Total unique variables: {len(df_presence)}')

In [ ]:
# Summary stats
print('=== DEMOGRAPHIC VARIABLES ===')
demo_only = df_presence[df_presence['type'] == 'demographic']
print(f'  In GSS columns: {demo_only["in_gss"].sum()} / {len(demo_only)}')
print(f'  Present in all 3 years: {demo_only["present_all_years"].sum()}')
print(f'  Present in 2 years: {(demo_only["n_years_present"] == 2).sum()}')
print(f'  Present in 1 year: {(demo_only["n_years_present"] == 1).sum()}')
print(f'  Present in 0 years: {(demo_only["n_years_present"] == 0).sum()}')

print('\n=== LIFESTYLE VARIABLES ===')
life_only = df_presence[df_presence['type'] == 'lifestyle']
print(f'  In GSS columns: {life_only["in_gss"].sum()} / {len(life_only)}')
print(f'  Present in all 3 years: {life_only["present_all_years"].sum()}')
print(f'  Present in 2 years: {(life_only["n_years_present"] == 2).sum()}')
print(f'  Present in 1 year: {(life_only["n_years_present"] == 1).sum()}')
print(f'  Present in 0 years: {(life_only["n_years_present"] == 0).sum()}')

In [ ]:
# Show demographic variables present in all 3 years, sorted by average coverage
demo_all3 = demo_only[demo_only['present_all_years']].copy()
pct_cols = [f'pct_{y}' for y in years]
demo_all3['avg_pct'] = demo_all3[pct_cols].mean(axis=1)
demo_all3 = demo_all3.sort_values('avg_pct', ascending=False)

print(f'Demographic variables present in all 3 years ({len(demo_all3)}):')
print(demo_all3[['variable', *pct_cols, 'avg_pct']].to_string(index=False))

In [ ]:
# Show lifestyle variables present in all 3 years
life_all3 = life_only[life_only['present_all_years']].copy()
life_all3['avg_pct'] = life_all3[pct_cols].mean(axis=1)
life_all3 = life_all3.sort_values('avg_pct', ascending=False)

print(f'Lifestyle variables present in all 3 years ({len(life_all3)}):')
print(life_all3[['variable', *pct_cols, 'avg_pct']].to_string(index=False))

In [ ]:
# Show variables NOT present in all 3 years
not_all3 = df_presence[~df_presence['present_all_years']].sort_values('n_years_present')
print(f'Variables NOT in all 3 years ({len(not_all3)}):')
print(not_all3[['variable', 'type', *pct_cols, 'n_years_present']].to_string(index=False))

## Part 2: Correlation Analysis — Public Issues (demo sim results)

In [ ]:
# Load LLM results
df_llm = pd.read_pickle('llm_results/df_demo_sim_public_issues_Llama-3.1-8B-Instruct_20260206_024540.pkl')
print(f'LLM results: {len(df_llm)} topics')
print(df_llm.columns.tolist())
df_llm.head()

In [ ]:
# Load GSS survey polarization
pol_pub = pd.read_csv('public_issues_polarization.csv')
print(f'Polarization data: {len(pol_pub)} topics')

# Merge
df_merged = df_llm.merge(
    pol_pub[['variable', 'polarization', 'area']].rename(
        columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
    ),
    on='Topic', how='inner'
)
print(f'Merged: {len(df_merged)} topics')
df_merged.head()

In [ ]:
# Correlation: LLM activation polarization vs GSS survey polarization
print('=' * 60)
print('CORRELATION: LLM Activation Polarization vs GSS Survey Polarization')
print('=' * 60)

for method in ['mean', 'median']:
    suffix = '' if method == 'mean' else '_median'
    col = f'Avg_Mahal_PCA15{suffix}'
    if col not in df_merged.columns:
        continue
    r_p, p_p = pearsonr(df_merged[col], df_merged['GSS_Polarization'])
    r_s, p_s = spearmanr(df_merged[col], df_merged['GSS_Polarization'])
    print(f'\n  Centroid: {method}')
    print(f'    Pearson:  r={r_p:.4f}  (p={p_p:.2e})')
    print(f'    Spearman: rho={r_s:.4f}  (p={p_s:.2e})')
    print(f'    N topics: {len(df_merged)}')

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, method in zip(axes, ['mean', 'median']):
    suffix = '' if method == 'mean' else '_median'
    col = f'Avg_Mahal_PCA15{suffix}'
    if col not in df_merged.columns:
        continue
    
    r_p, _ = pearsonr(df_merged[col], df_merged['GSS_Polarization'])
    r_s, _ = spearmanr(df_merged[col], df_merged['GSS_Polarization'])
    
    ax.scatter(df_merged['GSS_Polarization'], df_merged[col], alpha=0.6, s=30)
    
    # Trend line
    z = np.polyfit(df_merged['GSS_Polarization'], df_merged[col], 1)
    x_line = np.linspace(df_merged['GSS_Polarization'].min(), df_merged['GSS_Polarization'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', alpha=0.7)
    
    ax.set_xlabel('GSS Survey Polarization')
    ax.set_ylabel(f'LLM Activation Polarization (PCA-15, {method})')
    ax.set_title(f'Public Issues — {method} centroid\nr={r_p:.3f}, rho={r_s:.3f}, n={len(df_merged)}')

plt.tight_layout()
plt.show()

In [ ]:
# Filtered results (exclude problematic topics)
EXCLUDE_PUBLIC = {'hubbywk1', 'racdif1', 'racdif2', 'racdif3', 'racdif4',
                  'workwhts', 'wlthwhts', 'intlwhts'}

df_filtered = df_merged[~df_merged['Topic'].isin(EXCLUDE_PUBLIC)].reset_index(drop=True)
print(f'After excluding {len(EXCLUDE_PUBLIC)} topics: {len(df_filtered)} remain')

print('\n' + '=' * 60)
print('FILTERED CORRELATIONS')
print('=' * 60)

for method in ['mean', 'median']:
    suffix = '' if method == 'mean' else '_median'
    col = f'Avg_Mahal_PCA15{suffix}'
    if col not in df_filtered.columns:
        continue
    r_p, p_p = pearsonr(df_filtered[col], df_filtered['GSS_Polarization'])
    r_s, p_s = spearmanr(df_filtered[col], df_filtered['GSS_Polarization'])
    print(f'\n  Centroid: {method}')
    print(f'    Pearson:  r={r_p:.4f}  (p={p_p:.2e})')
    print(f'    Spearman: rho={r_s:.4f}  (p={p_s:.2e})')
    print(f'    N topics: {len(df_filtered)}')

In [ ]:
# Breakdown by topic area
print('\n' + '=' * 60)
print('CORRELATION BY TOPIC AREA')
print('=' * 60)

col = 'Avg_Mahal_PCA15'
for area in sorted(df_merged['area'].unique()):
    subset = df_merged[df_merged['area'] == area]
    if len(subset) < 3:
        continue
    r_p, _ = pearsonr(subset[col], subset['GSS_Polarization'])
    r_s, _ = spearmanr(subset[col], subset['GSS_Polarization'])
    print(f'  {area:50s}  n={len(subset):3d}  r={r_p:+.3f}  rho={r_s:+.3f}')

In [ ]:
# Show all topics sorted by LLM activation polarization
col = 'Avg_Mahal_PCA15'
df_show = df_merged[['Topic', 'area', 'n_valid', 'n_dem', 'n_rep',
                      'GSS_Polarization', col, f'{col}_median']].copy()
df_show = df_show.sort_values(col, ascending=False)
print(f'All {len(df_show)} topics sorted by LLM activation polarization (mean centroid):')
print(df_show.to_string(index=False))

In [ ]:
# Residual analysis: which topics are most over/under-predicted?
col = 'Avg_Mahal_PCA15'
# Standardize both columns for comparable residuals
x = df_merged['GSS_Polarization']
y = df_merged[col]
z = np.polyfit(x, y, 1)
predicted = np.polyval(z, x)
df_merged['residual'] = y - predicted

print('Topics with HIGHEST positive residual (LLM sees more polarization than survey):')
top_pos = df_merged.nlargest(10, 'residual')[['Topic', 'area', 'GSS_Polarization', col, 'residual']]
print(top_pos.to_string(index=False))

print('\nTopics with HIGHEST negative residual (LLM sees less polarization than survey):')
top_neg = df_merged.nsmallest(10, 'residual')[['Topic', 'area', 'GSS_Polarization', col, 'residual']]
print(top_neg.to_string(index=False))

## Part 3: PC1–Answer Correlation

For each topic, the updated `run_demo_sim.py` computes PC1 of each attention head's activations
and correlates it with the actual survey response (taking |r| since PC1 sign is arbitrary).

This tells us: **does the model's internal representation linearly track the actual answer?**

Requires re-running with the updated script that outputs `Avg_PC1_Answer_Corr` and `Max_PC1_Answer_Corr`.

In [ ]:
# Load results from re-run (update path as needed)
import glob

# Find the latest demo_sim public_issues pkl
pkl_files = sorted(glob.glob('llm_results/df_demo_sim_public_issues_Llama-3.1-8B-Instruct_*.pkl'))
latest_pkl = pkl_files[-1] if pkl_files else None
print(f'Latest pkl: {latest_pkl}')

df_llm2 = pd.read_pickle(latest_pkl)
print(f'Columns: {df_llm2.columns.tolist()}')

has_pc1 = 'Avg_PC1_Answer_Corr' in df_llm2.columns
print(f'Has PC1 correlation data: {has_pc1}')
if has_pc1:
    print(f'\nTopics: {len(df_llm2)}')
    print(f'Avg_PC1_Answer_Corr: mean={df_llm2["Avg_PC1_Answer_Corr"].mean():.4f}, '
          f'std={df_llm2["Avg_PC1_Answer_Corr"].std():.4f}')
    print(f'Max_PC1_Answer_Corr: mean={df_llm2["Max_PC1_Answer_Corr"].mean():.4f}, '
          f'std={df_llm2["Max_PC1_Answer_Corr"].std():.4f}')

In [ ]:
assert has_pc1, 'Re-run run_demo_sim.py with the updated script to get PC1 correlation data'

# Merge with survey polarization for context
df_pc1 = df_llm2.merge(
    pol_pub[['variable', 'polarization', 'area']].rename(
        columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
    ),
    on='Topic', how='inner'
)

# Sort by Avg PC1-answer correlation
df_pc1_sorted = df_pc1.sort_values('Avg_PC1_Answer_Corr', ascending=False)

print(f'All {len(df_pc1_sorted)} topics sorted by Avg |PC1-answer correlation|:')
print(df_pc1_sorted[['Topic', 'area', 'n_valid',
                      'Avg_PC1_Answer_Corr', 'Max_PC1_Answer_Corr',
                      'GSS_Polarization']].to_string(index=False))

In [ ]:
# Distribution of PC1-answer correlations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_pc1['Avg_PC1_Answer_Corr'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Avg |PC1-Answer Correlation| across heads')
axes[0].set_ylabel('Number of topics')
axes[0].set_title(f'Distribution of Avg PC1-Answer |r|\n(mean={df_pc1["Avg_PC1_Answer_Corr"].mean():.3f})')
axes[0].axvline(df_pc1['Avg_PC1_Answer_Corr'].mean(), color='red', linestyle='--', label='mean')
axes[0].legend()

axes[1].hist(df_pc1['Max_PC1_Answer_Corr'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Max |PC1-Answer Correlation| across heads')
axes[1].set_ylabel('Number of topics')
axes[1].set_title(f'Distribution of Max PC1-Answer |r|\n(mean={df_pc1["Max_PC1_Answer_Corr"].mean():.3f})')
axes[1].axvline(df_pc1['Max_PC1_Answer_Corr'].mean(), color='red', linestyle='--', label='mean')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Is PC1-answer correlation related to survey polarization?
# i.e., do more polarized questions also have activations that track the answer better?
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, col, label in zip(axes,
    ['Avg_PC1_Answer_Corr', 'Max_PC1_Answer_Corr'],
    ['Avg |PC1-Answer r|', 'Max |PC1-Answer r|']):

    r_p, p_p = pearsonr(df_pc1[col], df_pc1['GSS_Polarization'])
    r_s, p_s = spearmanr(df_pc1[col], df_pc1['GSS_Polarization'])

    ax.scatter(df_pc1['GSS_Polarization'], df_pc1[col], alpha=0.6, s=30)
    z = np.polyfit(df_pc1['GSS_Polarization'], df_pc1[col], 1)
    x_line = np.linspace(df_pc1['GSS_Polarization'].min(), df_pc1['GSS_Polarization'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', alpha=0.7)

    ax.set_xlabel('GSS Survey Polarization')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs Survey Polarization\nr={r_p:.3f} (p={p_p:.2e}), rho={r_s:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Is PC1-answer correlation related to activation polarization (Mahalanobis)?
col_mahal = 'Avg_Mahal_PCA15'
col_pc1 = 'Avg_PC1_Answer_Corr'

if col_mahal in df_pc1.columns:
    r_p, p_p = pearsonr(df_pc1[col_pc1], df_pc1[col_mahal])
    r_s, p_s = spearmanr(df_pc1[col_pc1], df_pc1[col_mahal])

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df_pc1[col_mahal], df_pc1[col_pc1], alpha=0.6, s=30)
    z = np.polyfit(df_pc1[col_mahal], df_pc1[col_pc1], 1)
    x_line = np.linspace(df_pc1[col_mahal].min(), df_pc1[col_mahal].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', alpha=0.7)
    ax.set_xlabel('LLM Activation Polarization (Avg Mahalanobis PCA-15)')
    ax.set_ylabel('Avg |PC1-Answer Correlation|')
    ax.set_title(f'Activation Polarization vs PC1-Answer Tracking\nr={r_p:.3f} (p={p_p:.2e}), rho={r_s:.3f}')
    plt.tight_layout()
    plt.show()

    print(f'Pearson:  r={r_p:.4f}  (p={p_p:.2e})')
    print(f'Spearman: rho={r_s:.4f}  (p={p_s:.2e})')

In [ ]:
# Breakdown by topic area: average PC1-answer correlation
print('PC1-Answer |r| by topic area:')
print('=' * 70)
area_stats = df_pc1.groupby('area').agg(
    n=('Avg_PC1_Answer_Corr', 'count'),
    avg_pc1_corr=('Avg_PC1_Answer_Corr', 'mean'),
    max_pc1_corr=('Max_PC1_Answer_Corr', 'mean'),
    avg_polarization=('GSS_Polarization', 'mean'),
).sort_values('avg_pc1_corr', ascending=False)
print(area_stats.to_string())